First of all, we init the SPARQLWrapper service with the SPARQL endpoint

In [8]:
!pip install SPARQLWrapper

In [15]:
import csv
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.error import HTTPError
import time

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader(
    "User-Agent",
    "MyResearchTool/1.0 (myemail@example.com)")
sparql.setReturnFormat(JSON)

Then we define our CONSTRUCT query to extract the metadata

In [16]:
sparql.setQuery("""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>

SELECT DISTINCT ?s ?sLabel ?author ?authorLabel 
  (SAMPLE(?date) AS ?year) 
  (GROUP_CONCAT(DISTINCT ?depicts; SEPARATOR="; ") AS ?depictsAll) 
  (GROUP_CONCAT(DISTINCT ?depictsTxt; SEPARATOR="; ") 
      AS ?depictLabels)  
WHERE {?s wdt:P8905 ?prado . 
       ?s wdt:P18 ?image.
       OPTIONAL {?s wdt:P180 ?depicts. 
           ?depicts rdfs:label ?depictsTxt . 
           FILTER (LANG(?depictsTxt) = "en")}
       ?s wdt:P571 ?date.
       ?s wdt:P170 ?author
    SERVICE wikibase:label { bd:serviceParam wikibase:language 
                               "[AUTO_LANGUAGE],mul,en". }
}
GROUP BY ?s ?sLabel ?author ?authorLabel ?year
"""
)

Finally, we serialise the result

In [ ]:
for attempt in range(5):
    try:
        results = sparql.query().convert()
        break

    except HTTPError as e:
        if e.code == 429:
            wait = 60 * (attempt + 1)
            print(f"Rate limited. Waiting {wait} seconds...")
            time.sleep(wait)
        else:
            raise

results = sparql.queryAndConvert()
print(results.serialize())

Rate limited. Waiting 60 seconds...
Rate limited. Waiting 120 seconds...
Rate limited. Waiting 180 seconds...


In [5]:
with open("output/prado.ttl", "w") as text_file:
    text_file.write(results.serialize())

#### References
- Candela, G. (2023). Towards a semantic approach in GLAM Labs: The case of the Data Foundry at the National Library of Scotland. Journal of Information Science, 52(1), 3–21. https://doi.org/10.1177/01655515231174386
- Dişli, M., Osti, G., Candela, G., & Zijdeman, R. (2025). From Linked Open Data to Collections as Data: A Reproducible Framework Using Federated Queries. Information Technology and Libraries, 44(4). https://doi.org/10.5860/ital.v44i4.17432